# 第 6 天：复杂动力学第一定律（The First Law of Complexodynamics）

> Scott Aaronson（2011）——[博文](https://scottaaronson.blog/?p=762)

**核心问题：**为什么熵趋于增加时，“有趣程度”会在中间时刻达到峰值？

**Aaronson 的答案：**复杂熵（complextropy；资源受限的精致复杂度 sophistication）被猜想在初期低、中间高、后期低。

本 Notebook 构建咖啡混合模拟，测量多种复杂性代理量。它是教学探索，不是复杂熵的精确计算或猜想的证明。


In [ ]:
import numpy as np
import gzip
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 1. 用 gzip 近似 KC
柯尔莫哥洛夫复杂度（Kolmogorov complexity，KC）不可计算。gzip 加固定解压开销可以给出描述长度上界；此处只记录压缩大小。


In [ ]:
def gzip_complexity(data: bytes) -> int:
    return len(gzip.compress(data, compresslevel=9))

zeros = bytes(1024)
random_data = np.random.randint(0, 256, 1024, dtype=np.uint8).tobytes()
pattern = b'ABCD' * 256

print(f'全零：{gzip_complexity(zeros)} 字节')
print(f'模式：{gzip_complexity(pattern)} 字节')
print(f'随机：{gzip_complexity(random_data)} 字节')

## 2. 咖啡混合模拟
二维像素网格：上半部分黑（咖啡），下半部分白（牛奶），随机尝试交换邻居。

实现说明：代码随机选择像素和方向，可能跳过越界或同色像素；它不同于每步均匀选择所有异色相邻对的模型。交换尝试次数不等于成功交换次数。


In [ ]:
def create_grid(size=64):
    grid = np.zeros((size, size), dtype=np.uint8)
    grid[:size//2, :] = 1
    return grid

def batch_swap(grid, n_swaps=100):
    rows, cols = grid.shape
    for _ in range(n_swaps):
        r, c = np.random.randint(rows), np.random.randint(cols)
        d = np.random.randint(4)
        if d == 0 and r > 0: nr, nc = r-1, c
        elif d == 1 and r < rows-1: nr, nc = r+1, c
        elif d == 2 and c > 0: nr, nc = r, c-1
        elif d == 3 and c < cols-1: nr, nc = r, c+1
        else: continue
        if grid[r, c] != grid[nr, nc]:
            grid[r, c], grid[nr, nc] = grid[nr, nc], grid[r, c]
    return grid

grid = create_grid(64)
plt.imshow(grid, cmap='gray', interpolation='nearest')
plt.title('初始状态：咖啡与牛奶分离')
plt.axis('off'); plt.show()

## 3. 观察混合
分离（简单）-> 卷须纹理（复杂）-> 粗粒度下均匀（简单）。微观二值网格后期仍是随机像素，不会变为真正的均匀灰度数组。


In [ ]:
np.random.seed(42)
grid = create_grid(64)
snapshots = [(0, grid.copy())]
n_steps = 800
for step in range(1, n_steps + 1):
    batch_swap(grid, 50)
    if step % 160 == 0:
        snapshots.append((step * 50, grid.copy()))

fig, axes = plt.subplots(1, len(snapshots), figsize=(3*len(snapshots), 3))
for ax, (t, g) in zip(axes, snapshots):
    ax.imshow(g, cmap='gray', interpolation='nearest')
    ax.set_title(f't={t:,}'); ax.axis('off')
fig.suptitle('咖啡混合过程'); plt.tight_layout(); plt.show()

## 4. 复杂性隆起
随时间跟踪 gzip 复杂度。原说明预期先增加、后减少；但原始 gzip 压缩大小并不保证呈现隆起，最大样本点本身也不能证明存在内部峰值。


In [ ]:
np.random.seed(42)
grid = create_grid(64)
times, complexities = [0], [gzip_complexity(grid.tobytes())]

for step in range(1, 1001):
    batch_swap(grid, 50)
    if step % 10 == 0:
        times.append(step * 50)
        complexities.append(gzip_complexity(grid.tobytes()))

peak_idx = np.argmax(complexities)
plt.plot(times, complexities, 'r-', lw=2)
plt.plot(times[peak_idx], complexities[peak_idx], 'ko', ms=10)
plt.xlabel('交换尝试总次数'); plt.ylabel('gzip 大小（字节）')
plt.title('复杂性隆起检验'); plt.grid(True, alpha=0.3); plt.show()
print(f'最大样本点的交换尝试次数：{times[peak_idx]:,}：{complexities[peak_idx]} 字节')

## 5. 熵与复杂性
Aaronson 试图形式化地刻画：熵趋于增加，而结构复杂性在中间时刻达到峰值。

下列代码计算平均局部像素熵，通常随混合上升，但单条随机轨迹不保证逐步单调。它不同于 `implementation.py` 中守恒的全局像素熵，也不是热力学熵的严格测量。


In [ ]:
def local_entropy(grid, w=8):
    h, width = grid.shape
    vals = []
    for i in range(0, h-w, w//2):
        for j in range(0, width-w, w//2):
            p = grid[i:i+w, j:j+w].mean()
            if 0 < p < 1:
                vals.append(-(p*np.log2(p)+(1-p)*np.log2(1-p)))
            else: vals.append(0.0)
    return np.mean(vals)

np.random.seed(42)
grid = create_grid(64)
times, kc_vals, ent_vals = [0], [gzip_complexity(grid.tobytes())], [local_entropy(grid)]
for step in range(1, 1001):
    batch_swap(grid, 50)
    if step % 10 == 0:
        times.append(step*50)
        kc_vals.append(gzip_complexity(grid.tobytes()))
        ent_vals.append(local_entropy(grid))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(times, ent_vals, 'b-', lw=2); axes[0].set_title('局部熵（通常上升，非严格单调）')
axes[1].plot(times, kc_vals, 'r-', lw=2); axes[1].set_title('复杂性（检验隆起）')
for ax in axes: ax.set_xlabel('交换尝试次数'); ax.grid(True, alpha=0.3)
plt.suptitle('熵与复杂性'); plt.tight_layout(); plt.show()

## 小结

- 平均局部熵通常随混合上升，但不保证逐步单调；全局像素熵因交换守恒比例而不变。
- gzip 大小是 KC 的粗略代理量，不保证在中间时刻出现隆起，也不等于复杂熵。
- Aaronson 猜想复杂熵应在中间时刻达到峰值；原仓库称截至 2024 年未证明，本次未独立核实最新状态。
- 第 7 天在咖啡自动机中对这些度量进行实证检验。
